# NB12 · Exceptions

`intermediate/` · 20 questions · the last one is a story project

An exception is Python's way of saying "I can't carry on from here".
You have met `ValueError`, `KeyError` and `ZeroDivisionError` already, and
NB07 showed a first `try` / `except`. This notebook goes deep: catching
the right errors, raising your own, chaining them and reporting several
at once.

Exceptions form a family tree (`KeyError` and `IndexError` are both kinds
of `LookupError`, which is a kind of `Exception`). An `except` clause
catches the class you name **and everything below it**. Custom exceptions
use the one-line form `class MyError(Exception): pass`; classes with
methods come in NB14.

Every solution here **catches and reports** its errors, so no cell ends in
a crash. To catch several classes in one clause, put them in a tuple:
`except (KeyError, IndexError):`.

## You'll need

- NB11: functions and recursion

## You'll practise

- The built-in exception hierarchy (LookupError above KeyError and IndexError)
- try / except / else / finally; several except clauses; as e
- raise, re-raise, raise ... from (exception chaining)
- Custom exceptions (the one-line class form; full classes come in NB14)
- EAFP vs LBYL
- assert, and why it isn't input validation
- ExceptionGroup and except* (3.11+), add_note()
- e.args, __cause__ vs __context__, the traceback module
- 3.14: except A, B without brackets; return in finally now warns

## How it works

Write your code in the cell under each question, run it, and compare with **Expected output**. Open **Solution** only after a real attempt.

Questions get longer as you go: one-liners first, a small story project last. Every solution only uses what this notebook and the ones before it teach.

---

## Warm-up · one line each

### Q01 · Real-life

A bill is split between friends. Print `total / people`, but when `people` is zero print `cannot divide by zero` instead of crashing.

**Expected output:**

```text
cannot divide by zero
```

<details><summary><b>Solution</b></summary>

```python
try:
    print(total / people)
except ZeroDivisionError:
    print("cannot divide by zero")
```

</details>

In [ ]:
total = 120
people = 0

# your code here


### Q02 · Real-life

A cinema customer asks for a seat number that may not exist. Print the seat at index `wanted`, or `no such seat` if the index is out of range.

**Expected output:**

```text
no such seat
```

<details><summary><b>Solution</b></summary>

```python
try:
    print(seats[wanted])
except IndexError:
    print("no such seat")
```

</details>

In [ ]:
seats = ["A1", "A2", "A3"]
wanted = 5

# your code here


### Q03 · Build

Look up `"pear"` in the price dictionary, catching the error through its **parent class** `LookupError` (not `KeyError`). Store the error as `e` and print the name of its actual class.

**Expected output:**

```text
KeyError
```

<details><summary><b>Solution</b></summary>

```python
try:
    print(prices["pear"])
except LookupError as e:
    print(type(e).__name__)
```

</details>

In [ ]:
prices = {"apple": 0.5, "banana": 0.25}

# your code here


### Q04 · Output-match

Convert `text` with `int()`. When it fails, import `traceback` and print the one-line summary from `traceback.format_exception_only(e)` (it returns a list; take its first item and `strip()` the newline).

**Expected output:**

```text
ValueError: invalid literal for int() with base 10: '4x'
```

<details><summary><b>Solution</b></summary>

```python
import traceback
try:
    int(text)
except ValueError as e:
    print(traceback.format_exception_only(e)[0].strip())
```

</details>

In [ ]:
text = "4x"

# your code here


### Q05 · Output-match

Convert `text` with `int()`. When it fails, print the error's `args` tuple (not the error itself).

**Expected output:**

```text
("invalid literal for int() with base 10: '4x'",)
```

<details><summary><b>Solution</b></summary>

```python
try:
    int(text)
except ValueError as e:
    print(e.args)
```

</details>

In [ ]:
text = "4x"

# your code here


---

## Combine · 2 to 4 lines

### Q06 · Real-life

A survey form contains mixed answers. Add up the answers that are whole numbers, skip (and count) the ones that are not, and print the total and the skipped count on separate lines.

**Expected output:**

```text
59
2
```

<details><summary><b>Solution</b></summary>

```python
total = 0
skipped = 0
for answer in answers:
    try:
        total += int(answer)
    except ValueError:
        skipped += 1
print(total)
print(skipped)
```

</details>

In [ ]:
answers = ["12", "x", "7", "3.5", "40"]

# your code here


### Q07 · Output-match

Divide 100 by each divisor. Use all four parts of the statement:
`try` does the division, `except ZeroDivisionError` prints `bad divisor`,
`else` prints `result` and the value, and `finally` prints `checked` and
the divisor (it runs every time).

**Expected output:**

```text
result 25.0
checked 4
bad divisor
checked 0
result 12.5
checked 8
```

<details><summary><b>Solution</b></summary>

```python
for divisor in divisors:
    try:
        result = 100 / divisor
    except ZeroDivisionError:
        print("bad divisor")
    else:
        print("result", result)
    finally:
        print("checked", divisor)
```

</details>

In [ ]:
divisors = [4, 0, 8]

# your code here


### Q08 · Puzzle

Write `read_sensor()` that prints `reading` and then does
`return "OK"` inside a `try`, with a `finally` block that prints
`cleanup`. Print the value of calling it, and see which line comes first.

**Expected output:**

```text
reading
cleanup
OK
```

<details><summary><b>Solution</b></summary>

```python
def read_sensor():
    try:
        print("reading")
        return "OK"
    finally:
        print("cleanup")
print(read_sensor())
```

</details>

In [ ]:
# your code here


### Q09 · Real-life

A file importer processes uploads. Write `import_file(name, size_kb)`:
in a `try` it raises `ValueError("empty file")` when the size is 0 and
`OverflowError("too big")` when it is above 500; otherwise nothing is
raised. Print `imported` in `else`, the error message in the matching
`except`, and `closed` + the name in `finally`. Call it for each upload.

**Expected output:**

```text
imported
closed a.csv
empty file
closed b.csv
too big
closed c.csv
```

<details><summary><b>Solution</b></summary>

```python
def import_file(name, size_kb):
    try:
        if size_kb == 0:
            raise ValueError("empty file")
        if size_kb > 500:
            raise OverflowError("too big")
    except ValueError as e:
        print(e)
    except OverflowError as e:
        print(e)
    else:
        print("imported")
    finally:
        print("closed", name)
for name, size_kb in uploads:
    import_file(name, size_kb)
```

</details>

In [ ]:
uploads = [("a.csv", 40), ("b.csv", 0), ("c.csv", 900)]

# your code here


### Q10 · Real-life

A profile form must reject unrealistic ages. Write `set_age(age)` that
raises `ValueError` with a message that includes the age when it is below
0 or above 120, and otherwise returns the age. Loop over the ages; print
`age set to` and the value for good ones, or the error message for bad
ones.

**Expected output:**

```text
age set to 30
age -5 is not realistic
age 200 is not realistic
```

<details><summary><b>Solution</b></summary>

```python
def set_age(age):
    if age < 0 or age > 120:
        raise ValueError(f"age {age} is not realistic")
    return age
for age in ages:
    try:
        print("age set to", set_age(age))
    except ValueError as e:
        print(e)
```

</details>

In [ ]:
ages = [30, -5, 200]

# your code here


---

## Small real tasks · 5 to 8 lines

### Q11 · Build

Write `get(container, key)` that returns `container[key]`, or the text
`missing` when the lookup fails because of a missing dict key **or** an
out-of-range list index. Use one `except` line with both classes in a
tuple. Print a hit and a miss for a dict, then a hit and a
miss for a list.

**Expected output:**

```text
12
missing
20
missing
```

<details><summary><b>Solution</b></summary>

```python
def get(container, key):
    try:
        return container[key]
    except (KeyError, IndexError):
        return "missing"
print(get(stock, "apple"))
print(get(stock, "pear"))
print(get(rows, 1))
print(get(rows, 5))
```

</details>

In [ ]:
stock = {"apple": 12}
rows = [10, 20]

# your code here


### Q12 · Real-life

Define `InsufficientFunds` as a custom exception (one-line class based on
`Exception`). Write `withdraw(balance, amount)` that raises it with a
message saying how much the balance is short by, or returns the new
balance. Process every withdrawal, printing `balance` and the new amount,
or `declined:` and the message.

**Expected output:**

```text
balance 70
declined: short by 20
balance 20
```

<details><summary><b>Solution</b></summary>

```python
class InsufficientFunds(Exception): pass

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFunds(f"short by {amount - balance}")
    return balance - amount

for amount in requests:
    try:
        balance = withdraw(balance, amount)
        print("balance", balance)
    except InsufficientFunds as e:
        print("declined:", e)
```

</details>

In [ ]:
balance = 100
requests = [30, 90, 50]

# your code here


### Q13 · Build

A low-level `ValueError` is not a helpful message for a shop owner.
Define `BadPrice(Exception)`. Write two functions that convert text to a
float and raise `BadPrice("price is not a number")` when it fails:
`strict(text)` raises it with `from e`, while `careless(text)` raises it
inside the `except` block without `from`. Call each with `"abc"` and print
its `__cause__` (the explicit link) and the name of its `__context__`
class (the automatic link).

**Expected output:**

```text
could not convert string to float: 'abc'
ValueError
None
ValueError
```

<details><summary><b>Solution</b></summary>

```python
class BadPrice(Exception): pass

def strict(text):
    try:
        return float(text)
    except ValueError as e:
        raise BadPrice("price is not a number") from e

def careless(text):
    try:
        return float(text)
    except ValueError:
        raise BadPrice("price is not a number")

for func in (strict, careless):
    try:
        func("abc")
    except BadPrice as e:
        print(e.__cause__)
        print(type(e.__context__).__name__)
```

</details>

In [ ]:
# your code here


### Q14 · Real-life

Check stock for each wanted item, printing the name and the quantity, or
the name and `not stocked` for unknown items. Do it twice with the same
output: first the **look before you leap** way (test with `in`), then the
**easier to ask forgiveness** way (`try` / `except KeyError`).

**Expected output:**

```text
apple 5
kiwi not stocked
pear 0
apple 5
kiwi not stocked
pear 0
```

<details><summary><b>Solution</b></summary>

```python
for item in wanted:
    if item in stock:
        print(item, stock[item])
    else:
        print(item, "not stocked")
for item in wanted:
    try:
        print(item, stock[item])
    except KeyError:
        print(item, "not stocked")
```

</details>

In [ ]:
stock = {"apple": 5, "pear": 0}
wanted = ["apple", "kiwi", "pear"]

# your code here


### Q15 · Real-life

Write `average(scores)` that starts with an `assert` that the list is not
empty (message `no scores`) and returns the average. Call it for both
lists; print the average, or the assertion message when it fails.

**Expected output:**

```text
80.0
no scores
```

<details><summary><b>Solution</b></summary>

```python
def average(scores):
    assert len(scores) > 0, "no scores"
    return sum(scores) / len(scores)
for scores in (class_a, class_b):
    try:
        print(average(scores))
    except AssertionError as e:
        print(e)
```

</details>

In [ ]:
class_a = [70, 80, 90]
class_b = []

# your code here


---

## Puzzles & builds · 8 to 15 lines

### Q16 · Real-life

A download helper must try again. Write `retry(func, attempts)` that calls
`func()` and returns its result. Each time `ConnectionError` is raised
it prints `attempt`, the number and the message. On the last attempt it
adds a note `gave up after N attempts` to the error and **re-raises** it
(a bare `raise`). Call `retry(flaky, 2)` and print the error's first note
when it escapes; then reset `calls["n"]` to 0 and print `retry(flaky, 3)`.

**Expected output:**

```text
attempt 1 failed: server busy
attempt 2 failed: server busy
gave up after 2 attempts
attempt 1 failed: server busy
attempt 2 failed: server busy
data received
```

<details><summary><b>Solution</b></summary>

```python
def retry(func, attempts):
    for attempt in range(1, attempts + 1):
        try:
            return func()
        except ConnectionError as e:
            print("attempt", attempt, "failed:", e)
            if attempt == attempts:
                e.add_note(f"gave up after {attempts} attempts")
                raise

try:
    print(retry(flaky, 2))
except ConnectionError as e:
    print(e.__notes__[0])
calls["n"] = 0
print(retry(flaky, 3))
```

</details>

In [ ]:
calls = {"n": 0}

def flaky():
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("server busy")
    return "data received"

# your code here


### Q17 · Puzzle

`risky(err)` raises whatever exception it is given. Call it for each
error in the list inside one `try` with four `except` clauses that print
the **most specific** label: `key` for `KeyError`, `lookup` for any other
`LookupError`, `math` for `ArithmeticError`, and `other` for everything
else. Order the clauses so every error gets its best label.

**Expected output:**

```text
key
lookup
math
other
```

<details><summary><b>Solution</b></summary>

```python
for err in errors:
    try:
        risky(err)
    except KeyError:
        print("key")
    except LookupError:
        print("lookup")
    except ArithmeticError:
        print("math")
    except Exception:
        print("other")
```

</details>

In [ ]:
def risky(err):
    raise err

errors = [KeyError("a"), IndexError(3), ZeroDivisionError("zero"), ValueError("v")]

# your code here


### Q18 · Puzzle

`stop()` simulates the user pressing Ctrl+C by raising
`KeyboardInterrupt`. Show three outcomes, each printing one line. (1)
Call `stop()` inside `except Exception`; when this does not catch it,
an outer `except KeyboardInterrupt` prints `only the outer handler saw it`.
(2) Call it inside `except BaseException as e` and print the class name.
(3) Call it inside `except (Exception, KeyboardInterrupt)` and print `both kinds caught`.

**Expected output:**

```text
only the outer handler saw it
KeyboardInterrupt
both kinds caught
```

<details><summary><b>Solution</b></summary>

```python
try:
    try:
        stop()
    except Exception:
        print("Exception caught it")
except KeyboardInterrupt:
    print("only the outer handler saw it")
try:
    stop()
except BaseException as e:
    print(type(e).__name__)
try:
    stop()
except (Exception, KeyboardInterrupt):
    print("both kinds caught")
```

</details>

In [ ]:
def stop():
    raise KeyboardInterrupt

# your code here


### Q19 · Real-life

A config file is checked before an app starts, and the user wants **every**
problem at once. Write `validate(config)` that collects errors in a list:
a `ValueError` for each value that is not valid (`port` must be digits;
`host` must not be empty) and a `KeyError("timeout")` if the key
`timeout` is missing. If the list is not empty it raises
`ExceptionGroup("config problems", errors)`. Handle the group with
`except* ValueError` (print `bad value:` and each message) and
`except* KeyError` (print `missing:` and each key).

**Expected output:**

```text
bad value: port must be digits
bad value: host is empty
missing: timeout
```

<details><summary><b>Solution</b></summary>

```python
def validate(config):
    errors = []
    if not config["port"].isdigit():
        errors.append(ValueError("port must be digits"))
    if config["host"] == "":
        errors.append(ValueError("host is empty"))
    if "timeout" not in config:
        errors.append(KeyError("timeout"))
    if errors:
        raise ExceptionGroup("config problems", errors)

try:
    validate(config)
except* ValueError as group:
    for e in group.exceptions:
        print("bad value:", e)
except* KeyError as group:
    for e in group.exceptions:
        print("missing:", e.args[0])
```

</details>

In [ ]:
config = {"port": "80a", "host": ""}

# your code here


---

## Q20 · Story project: Robust Record Parser

A school exports student records as comma-separated lines (name, age,
score). Some lines are broken. You are building a parser that never
crashes and reports every bad line with its line number.

**Run these cells first.** They set up the starting files and data for every step below.

In [ ]:
lines = [
    "Ada,36,91.5",
    "Grace,x,88",
    "Linus,29",
    "",
    "Alan,41,95",
    "Edsger,200,70",
    "Margaret,85,abc",
]


### Step 1

Define `RecordError(Exception)` and two children, `FieldCountError` and `BadValue` (one-line classes). Write `parse_line(line)` that splits on commas and returns a dict with `name`, `age` (int) and `score` (float). Raise `FieldCountError` when there are not exactly 3 fields, and `BadValue` when the age or score is not a number (raise it `from` the original error) or the age is outside 0 to 120. Print the result for the first line.

**Expected output:**

```text
{'name': 'Ada', 'age': 36, 'score': 91.5}
```

<details><summary><b>Solution</b></summary>

```python
class RecordError(Exception): pass
class FieldCountError(RecordError): pass
class BadValue(RecordError): pass

def parse_line(line):
    parts = line.split(",")
    if len(parts) != 3:
        raise FieldCountError(f"expected 3 fields, got {len(parts)}")
    name, age_text, score_text = parts
    try:
        age = int(age_text)
        score = float(score_text)
    except ValueError as e:
        raise BadValue(f"not a number in {line!r}") from e
    if not 0 <= age <= 120:
        raise BadValue(f"age {age} out of range")
    return {"name": name, "age": age, "score": score}

print(parse_line(lines[0]))
```

</details>

In [ ]:
# your code here


### Step 2

Write `parse_all(lines)` returning `(records, problems)`. Skip blank lines. Number the lines from 1; for each `RecordError`, add `(line_number, message)` to the problems. Print how many records and how many problems you got.

**Expected output:**

```text
2 records, 4 problems
```

<details><summary><b>Solution</b></summary>

```python
def parse_all(lines):
    records = []
    problems = []
    for number, line in enumerate(lines, 1):
        if line.strip() == "":
            continue
        try:
            records.append(parse_line(line))
        except RecordError as e:
            problems.append((number, str(e)))
    return records, problems

records, problems = parse_all(lines)
print(len(records), "records,", len(problems), "problems")
```

</details>

In [ ]:
# your code here


### Step 3

Print a report: one line per problem like `line 2: message`, then one line per good record with the name and score.

**Expected output:**

```text
line 2: not a number in 'Grace,x,88'
line 3: expected 3 fields, got 2
line 6: age 200 out of range
line 7: not a number in 'Margaret,85,abc'
Ada 91.5
Alan 95.0
```

<details><summary><b>Solution</b></summary>

```python
for number, message in problems:
    print(f"line {number}: {message}")
for record in records:
    print(record["name"], record["score"])
```

</details>

In [ ]:
# your code here


### Step 4

Look at the chain behind a bad value. Parse `lines[1]` inside a `try`; when `BadValue` is raised, print its message, then the class name of its `__cause__` and the cause itself.

**Expected output:**

```text
not a number in 'Grace,x,88'
ValueError invalid literal for int() with base 10: 'x'
```

<details><summary><b>Solution</b></summary>

```python
try:
    parse_line(lines[1])
except BadValue as e:
    print(e)
    print(type(e.__cause__).__name__, e.__cause__)
```

</details>

In [ ]:
# your code here


### Step 5

Write `check_all(lines)` that tries every non-blank line, adds the note `line N` to each `RecordError` (with `add_note`), and raises `ExceptionGroup("bad lines", errors)` if there were any. Handle it with `except* FieldCountError` (print `count:`, the message and the first note) and `except* BadValue` (print `value:`, the message and the first note).

**Expected output:**

```text
count: expected 3 fields, got 2 line 3
value: not a number in 'Grace,x,88' line 2
value: age 200 out of range line 6
value: not a number in 'Margaret,85,abc' line 7
```

<details><summary><b>Solution</b></summary>

```python
def check_all(lines):
    errors = []
    for number, line in enumerate(lines, 1):
        if line.strip() == "":
            continue
        try:
            parse_line(line)
        except RecordError as e:
            e.add_note(f"line {number}")
            errors.append(e)
    if errors:
        raise ExceptionGroup("bad lines", errors)

try:
    check_all(lines)
except* FieldCountError as group:
    for e in group.exceptions:
        print("count:", e, e.__notes__[0])
except* BadValue as group:
    for e in group.exceptions:
        print("value:", e, e.__notes__[0])
```

</details>

In [ ]:
# your code here


---

## Done

Next up: **NB13 · Modules Files JSON CSV** in `intermediate/`.